In [ ]:
import os
os.environ["PYTHONWARNINGS"] = "ignore:pkg_resources is deprecated as an API:UserWarning"

import warnings
import random

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import brier_score_loss
from sklearn.pipeline import Pipeline
import quantstats as qs
import pandas as pd
import numpy as np

from src.helpers.model_matrix import align_and_fill_dates_across_tickers
from src.helpers.model_matrix import build_model_matrix_from_wrds
from src.helpers._extract import ensure_dir

warnings.filterwarnings(
    "ignore",
    message=r"pkg_resources is deprecated as an API\..*",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message=r".*pkg_resources is deprecated as an API.*",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module=r"pkg_resources(\.|$)"
)

warnings.filterwarnings(
    "ignore",
    message="Mean of empty slice",
    category=RuntimeWarning,
)

### 1. Data build & cleaning (CRSP/DSF/IBES/FF)

In [ ]:
tickers_list = [
    'AAPL', 'NVDA', 'MSFT', 'AMZN', 'TSLA',
    'GOOGL', 'LLY', 'WMT', 'JPM', 'BRK-B',
    # 'V', 'MA', 'XOM', 'ORCL', 'UNH', 'COST', 'PG', 'HD', 'NFLX',
    # 'JNJ', 'BAC', 'CRM', 'QQQ', 'ABBV', 'KO', 'CVX', 'TMUS', 'MRK', 'CSCO',
    # 'WFC', 'ACN', 'NOW', 'TSM', 'AXP', 'PEP', 'MCD', 'IBM', 'MS', 'DIS',
    # 'TMO', 'ABT', 'AMD', 'ADBE', 'PM', 'ISRG', 'GE', 'GS', 'INTU', 'CAT',
    # 'TXN', 'QCOM', 'RY', 'VZ', 'DHR', 'BKNG', 'T', 'BLK', 'SPGI',
    # 'RTX', 'PFE', 'NEE', 'HON', 'CMCSA', 'PGR', 'AMGN', 'LOW', 'ANET', 'UNP',
    # 'SYK', 'TJX', 'C', 'BA', 'SCHW', 'BSX', 'KKR', 'ETN',
    # 'COP', 'BX', 'PANW', 'ADP'
]

# will extract every possible ticker data (not just tickers_list) from wrds.
# the only wrds filters used are 'start' and 'end' dates.
# by extracting all possible ticker data, we can update tickers_list without
# connecting to wrds.
# the data are extracted from dsf, crsp, ff, and ibes (see src/migrations folder)
all_stocks = build_model_matrix_from_wrds(
    wrds_user="navid_namazi",
    start="2016-01-01",
    end="2021-01-01",
    chunk_size=500_000,
    tickers=tickers_list,
    use_run="last"  # "new", "last", or a specific folder name (i.e. "run_20250914_133747")
)

# ensure all stocks have the same date coverage
df = align_and_fill_dates_across_tickers(all_stocks=all_stocks)

### 2. Train/test split & rolling CV

In [ ]:
random_state = 42
random.seed(random_state)

# get all unique dates
dates_all = df.index.get_level_values("date").unique().sort_values()
trading_days_count = len(dates_all)
n_rows = df.shape[0]

print("Total Data:")
print(f"  Dates: {trading_days_count} trading days")
print(f"  Period: {dates_all.min().date()} to {dates_all.max().date()}")
print(f"  Rows: {n_rows:,} (stocks × dates)")

# split: in-sample (80%) vs out-of-sample (20%)
split_pct = 0.80
split_idx = int(trading_days_count * split_pct)

# in-sample: used for feature selection, hyperparameter tuning, model development
ins_dates = dates_all[:split_idx]

# out-of-sample: never touched until final evaluation
dates_out_sample = dates_all[split_idx:]

# split date (boundary between in-sample and out-of-sample)
split_date = dates_out_sample[0]

print("\nData Split:")
print("   In-Sample (Development Set):")
print(f"   Period: {ins_dates.min().date()} to {ins_dates.max().date()}")
print(f"   Dates: {len(ins_dates)} days ({len(ins_dates) / trading_days_count * 100:.1f}%)")
print("   Purpose: feature selection, hyperparameter tuning, rolling CV")

print("\nOut-Of-Sample (Test Set):")
print(f"   Period: {dates_out_sample.min().date()} to {dates_out_sample.max().date()}")
print(f"   Dates: {len(dates_out_sample)} days ({len(dates_out_sample) / trading_days_count * 100:.1f}%)")
print("   Purpose: final performance evaluation only")

print(f"\nSplit Date: {split_date.date()}")

In [ ]:
# Rolling window size configuration for in-sample (60/20/20 Split)
# When naming variables, ins short for in-sample, oos short for out-of-sample 

ins_window_size = len(ins_dates)
ins_training_window_size = int(0.6 * ins_window_size)
ins_validation_window_size = int(0.2 * ins_window_size)
target_folds_count = 10

# Calculate step size to achieve target number of folds
# Formula: step = (remaining_data) / (target_folds - 1)
remaining_data = ins_window_size - ins_training_window_size - ins_validation_window_size
step_size = max(1, remaining_data // max(1, target_folds_count - 1))

# Calculate actual number of windows
actual_folds = (ins_window_size - ins_training_window_size - ins_validation_window_size) // step_size + 1

print("Rolling Window Configuration (In-Sample Only):")
print(
    f"   Training window: {ins_training_window_size} days (~{ins_training_window_size / 252:.1f} years, {ins_training_window_size / ins_window_size * 100:.1f}% of in-sample)")
print(
    f"   Validation window: {ins_validation_window_size} days (~{ins_validation_window_size / 252:.1f} years, {ins_validation_window_size / ins_window_size * 100:.1f}% of in-sample)")
print(f"   Step size: {step_size} days (~{step_size / 5:.1f} weeks)")
print(f"   Target folds: {target_folds_count}")
print(f"   Actual folds: {actual_folds}")
print(f"   Coverage: {actual_folds * ins_validation_window_size} validation days out of {ins_window_size} total")

### 3. Hyperparameter tuning: logistic regression + ElasticNet

In [ ]:
# Binary Target Definition
print("Target Column: adjclose_lead = next-day log return(t -> t+1)")
print("We predict: will the stock go up (1) or down (0) tomorrow?")

DIR_binary = (df["adjclose_lead"] > 0).astype(int)  # 1 = up, 0 = down

# check class balance (market neutrality ~ 50/50)
print("\nBinary Target Distribution")
print(f" Up (1):   {(DIR_binary == 1).sum():,} observations ({(DIR_binary == 1).mean() * 100:.1f}%)")  # noqa
print(f" Down (0): {(DIR_binary == 0).sum():,} observations ({(DIR_binary == 0).mean() * 100:.1f}%)")  # noqa
print(f" Total:    {len(DIR_binary):,} observations")

# feature columns: everything except ticker, target, and the index columns, permno and date.
num_pred_cols = [c for c in df.columns if c not in (["ticker", "adjclose_lead"])]
print(f"\nUsing {len(num_pred_cols)} features for prediction")

In [ ]:
print("ElasticNet Hyperparameters and Pipeline Configuration")

# Default hyperparameters if tuning is skipped
ELASTICNET_C = 0.1
ELASTICNET_L1_RATIO = 0.7

print("\nHyperparameters:")
print(f"  C: {ELASTICNET_C}")
print(f"  l1_ratio: {ELASTICNET_L1_RATIO}")

# Preprocessing: Standardize features
ct = ColumnTransformer([(
    "num",  # numerical
    # scales each feature so ElasticNet penalty treats them similarly (to avoid leakage).
    # scaling is done per fold-by-fold
    StandardScaler(with_mean=True),
    num_pred_cols  # only numerical columns
)],
    remainder="drop",  # columns not listed are dropped
    sparse_threshold=0.0  # force dense for feature importance
)

# Classifier with ElasticNet regularization
clf = LogisticRegression(
    penalty="elasticnet",  # Use ElasticNet (L1 + L2)
    solver="saga",  # Only solver supporting elasticnet
    l1_ratio=ELASTICNET_L1_RATIO,  # Mix of L1 and L2
    C=ELASTICNET_C,  # Regularization strength
    max_iter=5000,  # More iterations for convergence
    tol=1e-4,
    random_state=random_state,
    n_jobs=-1  # Use all CPUs
)

# Pipeline: preprocessing to classification
pipe = Pipeline([("prep", ct), ("clf", clf)])

print("\nPipeline Flow:")
print("   1) 'prep' = ColumnTransformer(ct): standardize features (fit on train only to avoid leakage")
print("   2) 'clf'  = LogisticRegression with ElasticNet: learns β per fold")

# hyperparameter grid
param_grid = {
    "clf__C": [ELASTICNET_C * 0.5, ELASTICNET_C, ELASTICNET_C * 2.0],
    "clf__l1_ratio": [
        max(0.0, ELASTICNET_L1_RATIO - 0.2),
        ELASTICNET_L1_RATIO,
        min(1.0, ELASTICNET_L1_RATIO + 0.2),
    ],
}

### 4. Rolling training & feature selection per fold

In [ ]:
print("Rolling Window Training With Feature Selection (In-Sample Only)")

pred_prob_up_new = pd.Series(index=df.index, dtype=float)
pred_prob_down_new = pd.Series(index=df.index, dtype=float)
pred_score_new = pd.Series(index=df.index, dtype=float)
pred_class_new = pd.Series(index=df.index, dtype=int)
used_mask_new = pd.Series(False, index=df.index)

# track feature selection across windows
feature_selection_history = []

window_num = 0
start_pos = 0

while start_pos + ins_training_window_size + ins_validation_window_size <= len(ins_dates):
    window_num += 1

    train_dates = ins_dates[start_pos: start_pos + ins_training_window_size]
    valid_dates = ins_dates[
                  start_pos + ins_training_window_size: start_pos + ins_training_window_size + ins_validation_window_size]

    print(f"\nWindow {window_num}: {train_dates.min().date()} to {valid_dates.max().date()}")

    start_pos += step_size

    idx_tr = df.index.get_level_values("date").isin(train_dates)
    idx_va = df.index.get_level_values("date").isin(valid_dates)

    X_tr = df.loc[idx_tr, num_pred_cols]
    y_tr = DIR_binary.loc[idx_tr]
    X_va = df.loc[idx_va, num_pred_cols]
    y_va = DIR_binary.loc[idx_va]

    # tune on train only using Brier score (lower is better)
    gs = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="neg_brier_score",  # maximize -Brier == minimize Brier
        cv=3,
        n_jobs=-1,
        refit=True
    )
    gs.fit(X_tr, y_tr)
    pipe_best = gs.best_estimator_

    # validation RMSE report (probability RMSE = sqrt(Brier))
    p_up_val = pipe_best.predict_proba(X_va)[:, 1]
    rmse_val = np.sqrt(brier_score_loss(y_va, p_up_val))
    print(f"   Tuning best params: C={gs.best_params_['clf__C']}, l1={gs.best_params_['clf__l1_ratio']}")
    print(f"   Validation RMSE (proba): {rmse_val:.4f}")

    # feature Selection Analysis (from tuned model)
    coef = pipe_best.named_steps["clf"].coef_[0]

    feature_importance = pd.DataFrame({
        "feature": num_pred_cols,
        "coefficient": coef,
        "abs_coefficient": np.abs(coef)
    })

    ZERO_THRESHOLD = 1e-5
    selected_features = feature_importance[feature_importance["abs_coefficient"] > ZERO_THRESHOLD]
    n_selected = len(selected_features)
    pct_selected = (n_selected / len(num_pred_cols)) * 100

    top_features = selected_features.nlargest(10, "abs_coefficient")

    feature_selection_history.append({
        "window": window_num,
        "n_features_selected": n_selected,
        "pct_selected": pct_selected,
        "selected_features": selected_features["feature"].tolist(),
        "top_5_features": top_features.head(5)["feature"].tolist(),
        "train_start": train_dates.min(),
        "valid_end": valid_dates.max()
    })

    # predictions from the tuned model
    proba = pipe_best.predict_proba(X_va)
    p_down = proba[:, 0]
    p_up = proba[:, 1]
    score = 2 * p_up - 1
    yhat = (p_up > 0.5).astype(int)

    # store predictions
    pred_prob_up_new.loc[idx_va] = p_up
    pred_prob_down_new.loc[idx_va] = p_down
    pred_score_new.loc[idx_va] = score
    pred_class_new.loc[idx_va] = yhat
    used_mask_new.loc[idx_va] = True

    # report
    accuracy = (yhat == y_va).mean()  # noqa
    print(f"Training: {len(X_tr):,} samples")
    print(f"Validation: {len(X_va):,} samples, Accuracy: {accuracy:.1%}")
    print("\nFeature Selection:")
    print(f"   Selected: {n_selected}/{len(num_pred_cols)} features ({pct_selected:.1f}%)")
    print("   Top 5 features by importance:")
    for i, row in top_features.head(5).iterrows():
        print(f"      {i + 1}. {row['feature']}: {row['coefficient']:+.4f}")

print(f"\nTraining complete: {window_num} windows processed")
print(f"Total validated: {used_mask_new.sum():,} / {len(df):,}")

# update global predictions with new ones
pred_prob_up = pred_prob_up_new
pred_prob_down = pred_prob_down_new
pred_score = pred_score_new
pred_class = pred_class_new
used_mask = used_mask_new

### 6. Feature selection across all folds

In [ ]:
print("Feature Selection For Final Model")

# extract selection information from all windows
n_windows = len(feature_selection_history)
feature_selected_counts = {feat: 0 for feat in num_pred_cols}

# count how many windows each feature was selected in
for window_info in feature_selection_history:
    selected_features = window_info["selected_features"]
    for feat in selected_features:
        feature_selected_counts[feat] += 1

# calculate frequency (percentage of windows)
feature_freq = pd.DataFrame({
    "feature": list(feature_selected_counts.keys()),
    "count": list(feature_selected_counts.values()),
    "frequency": [count / n_windows for count in feature_selected_counts.values()]
}).sort_values("frequency", ascending=False)

SELECTION_THRESHOLD = 0.5  # moderate

selected_features_mask = feature_freq["frequency"] >= SELECTION_THRESHOLD
final_feature_list = feature_freq.loc[selected_features_mask, "feature"].tolist()

print("\n  Selection Criterion:")
print(f"   Frequency threshold: ≥ {SELECTION_THRESHOLD * 100:.0f}% of windows")
print("   Rationale: Features must be consistently predictive across")
print("              different market regimes to be included")

print("\n  Results:")
print(f"   Features selected: {len(final_feature_list)} / {len(num_pred_cols)}")
print(f"   Features removed:  {len(num_pred_cols) - len(final_feature_list)}")
print(f"   Reduction: {(1 - len(final_feature_list) / len(num_pred_cols)) * 100:.1f}%")

# display selected features
print(f"\n  Selected Features ({len(final_feature_list)}):")
print(f"    {'Feature':<30} {'Frequency':<12}     {'Appearances':<15}")

selected_features_df = feature_freq[selected_features_mask].copy()
for idx, row in selected_features_df.iterrows():
    feat = row['feature']
    freq = row['frequency']
    count = row['count']
    bar = '█' * int(freq * 10)
    print(f"    {feat:<30} {freq:>6.1%} ({count:>2}/{n_windows})   {bar}")

# display removed features
removed_features_df = feature_freq[~selected_features_mask].copy()

if len(removed_features_df) > 0:
    print(f"\n  Removed Features ({len(removed_features_df)}):")
    print(f"    {'Feature':<30} {'Frequency':<12}     {'Appearances':<15}")

    for idx, row in removed_features_df.iterrows():
        feat = row['feature']
        freq = row['frequency']
        count = row['count']
        bar = '░' * int(freq * 10)
        print(f"    {feat:<30} {freq:>6.1%} ({count:>2}/{n_windows})   {bar}")


# feature categories breakdown
def categorize_feature(feat):
    """Categorize feature by type"""
    if feat.startswith("ti_"):
        return "Technical Indicators"
    elif feat in ["mktrf", "smb", "hml", "rf", "umd"]:
        return "Fama-French Factors"
    elif feat.startswith("cons_") or feat.startswith("n_"):
        return "IBES Consensus"
    elif feat.startswith("adjclose_lag"):
        return "Price Lags"
    elif feat in ["adj_mktcap", "vol", "retx"]:
        return "Market Data"
    else:
        return "Other"


selected_features_df["category"] = selected_features_df["feature"].apply(categorize_feature)
category_counts = selected_features_df["category"].value_counts()

print("\n  Selected Features By Category:")

for category, count in category_counts.items():
    pct = count / len(selected_features_df) * 100
    print(f"    {category:<30} {count:>2} features ({pct:>5.1f}%)")

### 7. Train the final in-sample model on all in-sample data

In [ ]:
# hyperparameter selection on all in-sample

print("\nGlobal Hyperparameter Search (in-sample, using selected features)")

# build in-sample slice with final features
ins_mask = df.index.get_level_values("date").isin(ins_dates)
X_ins = df.loc[ins_mask, final_feature_list]
y_ins = DIR_binary.loc[ins_mask]

# group by date so folds split by day (avoid same-day leakage across stocks)
groups_ins = df.loc[ins_mask].index.get_level_values("date")

# rebuild pipeline constrained to the selected features
ct_final_cv = ColumnTransformer(
    [("num", StandardScaler(with_mean=True), final_feature_list)],
    remainder="drop",
    sparse_threshold=0.0
)
clf_final_cv = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    max_iter=5000,
    tol=1e-4,
    random_state=random_state,
    n_jobs=-1
)
pipe_final_cv = Pipeline([("prep", ct_final_cv), ("clf", clf_final_cv)])

param_grid_global = {
    "clf__C": [ELASTICNET_C * 0.5, ELASTICNET_C, ELASTICNET_C * 2.0],
    "clf__l1_ratio": [
        max(0.0, ELASTICNET_L1_RATIO - 0.2),
        ELASTICNET_L1_RATIO,
        min(1.0, ELASTICNET_L1_RATIO + 0.2),
    ],
}

# time-aware CV via GroupKFold over dates
cv_global = GroupKFold(n_splits=5)

gs_global = GridSearchCV(
    estimator=pipe_final_cv,
    param_grid=param_grid_global,
    scoring="neg_brier_score",  # minimize Brier (probability RMSE)
    cv=cv_global,
    n_jobs=-1,
    refit=True
)
gs_global.fit(X_ins, y_ins, groups=groups_ins)

best_C = gs_global.best_params_["clf__C"]
best_l1 = gs_global.best_params_["clf__l1_ratio"]
print(f"Best global params: C={best_C}, l1_ratio={best_l1}")
print(f"Best mean CV Brier: {-gs_global.best_score_:.6f} (RMSE={(-gs_global.best_score_) ** 0.5:.4f})")

# lock in the global best hyperparameters and train final in-sample model
clf_final = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    C=best_C,
    l1_ratio=best_l1,
    max_iter=5000,
    tol=1e-4,
    random_state=random_state,
    n_jobs=-1
)
ct_final = ColumnTransformer(
    [("num", StandardScaler(with_mean=True), final_feature_list)],
    remainder="drop",
    sparse_threshold=0.0
)
pipe_final = Pipeline([("prep", ct_final), ("clf", clf_final)])
pipe_final.fit(X_ins, y_ins)

# report sparsity of the final model
final_coef = pipe_final.named_steps["clf"].coef_[0]
print(f"Final model non-zero coeffs: {(np.abs(final_coef) > 1e-5).sum()} / {len(final_coef)}")

### 1. Trading Strategy (Long-Short Market Neutral) 

In [ ]:
# ranking helper
def ranking_equal_weights(scores, long_pct: float, short_pct: float) -> pd.Series:
    n = len(scores)
    n_long = max(1, int(n * long_pct))
    n_short = max(1, int(n * short_pct))
    idx = scores.sort_values(ascending=False).index
    w = pd.Series(0.0, index=scores.index)
    w.loc[idx[:n_long]] = 1.0 / n_long
    w.loc[idx[-n_short:]] = -1.0 / n_short
    return w


# volatility helpers
def rolling_annualized_vol(series: pd.Series, window: int = 20) -> pd.Series:
    # std of daily log-returns, then annualize
    vol = series.rolling(window=window, min_periods=max(1, window // 2)).std()
    return vol * np.sqrt(252)


def attach_volatility(
        frame: pd.DataFrame,
        ret_col: str = "adjclose_lead",
        idx_stock: str = "permno",
        idx_date: str = "date",
        window: int = 20,
        fill: str = "cross_section_median"
) -> pd.Series:
    # per-stock rolling vol
    vol = frame.groupby(level=idx_stock, group_keys=False)[ret_col].apply(
        lambda s: rolling_annualized_vol(s, window)
    )
    # cross-sectional fill
    if fill == "cross_section_median":
        vol = vol.groupby(level=idx_date).transform(lambda x: x.fillna(x.median()))
    return vol


# weighting transforms
def inverse_vol_adjust(raw_w: pd.Series, vol: pd.Series) -> pd.Series:
    return raw_w / vol


def normalize_dollar_neutral(
        w: pd.Series,
        long_target: float = 0.5,
        short_target: float = -0.5
) -> pd.Series:
    pos = w > 0
    neg = w < 0
    long_sum = w[pos].sum()
    short_sum = -w[neg].sum()
    out = w.copy()
    if long_sum > 0:
        out[pos] = w[pos] / long_sum * long_target
    if short_sum > 0:
        out[neg] = w[neg] / short_sum * (-short_target)
    return out


# performance helpers
def compute_simple_returns(logret: pd.Series) -> pd.Series:
    return np.exp(logret) - 1.0


def portfolio_returns(weights: pd.Series,
                      logret: pd.Series,
                      idx_date: str = "date") -> pd.Series:
    # weights are per (permno, date); multiply and sum by date
    daily = (weights * logret).groupby(level=idx_date).sum()
    return daily


def equity_from_simple_returns(simple_daily: pd.Series) -> pd.Series:
    return (1 + simple_daily).cumprod()


def stats_from_returns(simple_daily: pd.Series) -> dict:
    mu = simple_daily.mean()
    sd = simple_daily.std(ddof=0)
    sharpe = (mu / sd) * np.sqrt(252) if sd > 0 else np.nan
    eq = equity_from_simple_returns(simple_daily)
    total_ret = (eq.iloc[-1] - 1) * 100
    max_dd = (eq / eq.cummax() - 1).min() * 100
    return dict(
        trading_days=len(simple_daily),
        total_return=total_ret,
        sharpe=sharpe,
        max_drawdown=max_dd,
        avg_daily_ret=mu * 100,
        daily_vol=sd * 100,
        equity=eq
    )

In [ ]:
def build_long_short_portfolio(
        frame: pd.DataFrame,
        score_col: str,
        ret_col: str = "adjclose_lead",
        idx_date: str = "date",
        idx_stock: str = "permno",
        long_pct: float = 0.20,
        short_pct: float = 0.20,
        vol_window: int = 20,
        long_target: float = 0.5,
        short_target: float = -0.5,
        example_date_idx: int | None = 0,
        label: str = "Portfolio"
):
    # 1) ranking (equal-weight within buckets) per date
    w_raw = frame.groupby(level=idx_date, group_keys=False)[score_col] \
        .apply(lambda s: ranking_equal_weights(s, long_pct, short_pct))

    # 2) attach volatility and inverse-vol adjust
    vol = attach_volatility(frame, ret_col=ret_col, idx_stock=idx_stock, idx_date=idx_date, window=vol_window)
    w_vol = inverse_vol_adjust(w_raw, vol)

    # 3) normalize to dollar neutral per date
    w = w_vol.groupby(level=idx_date, group_keys=False).apply(
        lambda g: normalize_dollar_neutral(g, long_target=long_target, short_target=short_target)
    )

    # 4) diagnostics (optional prints)
    print(f"\nStrategy: {label} (top/bottom {int(long_pct * 100)}%)")
    print(f"  Total positions (non-zero): {(w != 0).sum():,}")  # noqa
    print(f"  Long positions:             {(w > 0).sum():,}")
    print(f"  Short positions:            {(w < 0).sum():,}")

    # one-day example
    if example_date_idx is not None:
        udates = frame.index.get_level_values(idx_date).unique()
        if 0 <= example_date_idx < len(udates):
            d = udates[example_date_idx]
            mask_d = frame.index.get_level_values(idx_date) == d
            w_d = w.loc[mask_d]
            print(f"\nExample for {pd.Timestamp(d).date()}:")
            print(f"  # longs:  {(w_d > 0).sum()} | sum longs:  {w_d[w_d > 0].sum():.4f}")
            print(f"  # shorts: {(w_d < 0).sum()} | sum shorts: {w_d[w_d < 0].sum():.4f}")
            print(f"  Net exposure: {w_d.sum():.4f}  (target ~ 0.00)")
            print(f"  Gross exposure: {w_d.abs().sum():.4f} (target ~ 1.00)")

    # 5) returns/metrics
    simple_daily = portfolio_returns(w, frame[ret_col], idx_date=idx_date)
    simple_daily = compute_simple_returns(simple_daily.apply(np.log1p))

    stats = stats_from_returns(simple_daily)
    print("\nPerformance:")
    print(f"  Trading days:  {stats['trading_days']}")
    print(f"  Total Return:  {stats['total_return']:7.2f}%")
    print(f"  Sharpe Ratio:  {stats['sharpe']:7.2f}")
    print(f"  Max Drawdown:  {stats['max_drawdown']:7.2f}%")
    print(f"  Avg Daily Ret: {stats['avg_daily_ret']:7.4f}%")
    print(f"  Daily Vol:     {stats['daily_vol']:7.4f}%")

    return dict(weights=w, vol=vol, daily_ret=simple_daily, equity=stats["equity"], stats=stats)

### 8. In-sample portfolio construction (Ranking-based trading strategy)

In [ ]:
ins_df = df.loc[used_mask].copy()
ins_df["score"] = pred_score.loc[used_mask]

ins_result = build_long_short_portfolio(
    frame=ins_df,
    score_col="score",
    ret_col="adjclose_lead",
    idx_date="date",
    idx_stock="permno",
    long_pct=0.20,
    short_pct=0.20,
    vol_window=20,
    long_target=0.5,
    short_target=-0.5,
    example_date_idx=10,
    label="In-sample (validation windows only)"
)

### 9. Out-of-sample evaluation & portfolio construction

In [ ]:
print("Out-Of-Sample Evaluation")

# define oos slice
oos_mask = df.index.get_level_values("date").isin(dates_out_sample)

# X_test: the features selected by the stability+ElasticNet process
# y_test: the binary target (Up=1, Down=0)
X_test = df.loc[oos_mask, final_feature_list]
y_test = DIR_binary.loc[oos_mask]

print("\nOut-of-sample period:")
print(f"  Dates: {dates_out_sample.min().date()} to {dates_out_sample.max().date()}")
print(f"  Days: {len(dates_out_sample)}")
print(f"  Samples: {len(X_test):,}")

# score with the frozen final model
# - predict_proba: probability of Up tomorrow
# - score: signed score in [-1, +1] (centered probability)
# - predict: class label (0/1)
test_prob_up = pipe_final.predict_proba(X_test)[:, 1]
test_prob_down = 1 - test_prob_up
test_score = 2 * test_prob_up - 1
test_pred = pipe_final.predict(X_test)

# classification summary
test_accuracy = (test_pred == y_test).mean()  # noqa

print("\nClassification Performance:")
print(f"  Accuracy: {test_accuracy:.1%}")
print(f"  Up class accuracy: {(test_pred[y_test == 1] == 1).mean():.1%}")  # noqa
print(f"  Down class accuracy: {(test_pred[y_test == 0] == 0).mean():.1%}")  # noqa

# confusion matrix + detailed classification report
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm, index=["Actual Down (0)", "Actual Up (1)"],
    columns=["Pred Down (0)", "Pred Up (1)"]
)
print("\nConfusion Matrix:")
print(cm_df)

report = classification_report(
    y_test,
    test_pred,
    labels=[0, 1],
    target_names=["Down (0)", "Up (1)"],
    zero_division=0
)
print("\nClassification Report")
print(report)

pred_prob_up_test = pd.Series(test_prob_up, index=X_test.index, name="prob_up")
pred_score_test = pd.Series(test_score, index=X_test.index, name="score")
pred_class_test = pd.Series(test_pred, index=X_test.index, name="pred_class")

print(f"\nTest predictions created for {len(pred_score_test):,} observations")
print("\nBuilding OOS Long-Short Portfolio (Top/Bottom 20%)")

oos_df = df.loc[oos_mask].copy()
oos_df["score"] = pred_score_test
oos_df["pred_class"] = pred_class_test
oos_df["prob_up"] = pred_prob_up_test

oos_result = build_long_short_portfolio(
    frame=oos_df,
    score_col="score",
    ret_col="adjclose_lead",
    idx_date="date",
    idx_stock="permno",
    long_pct=0.20,
    short_pct=0.20,
    vol_window=20,
    long_target=0.5,
    short_target=-0.5,
    example_date_idx=10,
    label="Out-of-sample"
)

stats = oos_result["stats"]
print("\nOOS Long-Short Headline Stats:")
print(f"  Total Return:   {stats.get('total_return', float('nan')):.2f}%")
print(f"  Sharpe Ratio:   {stats.get('sharpe', float('nan')):.2f}")
print(f"  Max Drawdown:   {stats.get('max_drawdown', float('nan')):.2f}%")
print(f"  Avg Daily Ret:  {stats.get('avg_daily_ret', float('nan')):.4f}%")
print(f"  Daily Vol:      {stats.get('daily_vol', float('nan')):.4f}%")

### 10. Generating Out-Of-Sample Reports

In [ ]:
print("Generating Out-Of-Sample HTML Reports")


def make_qs_report_from_equity(equity_series, rf_series, mktrf_series, title, out_path):
    # Daily simple returns from equity
    rets = equity_series.pct_change(fill_method=None).dropna()
    # Align RF & benchmark on the same index
    rf_aligned = rf_series.reindex(rets.index).ffill().bfill()
    mktrf_aln = mktrf_series.reindex(rets.index).ffill().bfill()
    bench_simple = (mktrf_aln + rf_aligned)
    # Excess returns
    strat_excess = (rets - rf_aligned).dropna()
    bench_excess = (bench_simple - rf_aligned).reindex(strat_excess.index).dropna()
    # Align both
    common_idx = strat_excess.index.intersection(bench_excess.index)
    strat_excess = strat_excess.reindex(common_idx)
    bench_excess = bench_excess.reindex(common_idx)

    qs.reports.html(
        strat_excess,
        benchmark=bench_excess.to_frame("Market"),
        rf=0.0,
        periods_per_year=252,
        output=out_path,
        title=title
    )
    print(f"    Saved: {out_path}")
    print(f"   Period: {strat_excess.index.min().date()} to {strat_excess.index.max().date()}")
    print(f"   Days:   {len(strat_excess)}")


# Prepare OOS Fama-French series
used_mask_oos = oos_result["weights"] != 0  # weights inside oos_result
rf_oos = (
    oos_df.loc[used_mask_oos]
    .reset_index()[["date", "rf"]]
    .dropna()
    .groupby("date", as_index=True)["rf"]
    .mean()
    .astype(float)
    .sort_index()
)
mktrf_oos = (
    oos_df.loc[used_mask_oos]
    .reset_index()[["date", "mktrf"]]
    .dropna()
    .groupby("date", as_index=True)["mktrf"]
    .mean()
    .astype(float)
    .sort_index()
)

# Generate HTML Reports: Out-Of-Sample Only (Both Strategies)
ensure_dir("out")
make_qs_report_from_equity(
    equity_series=oos_result["equity"],
    rf_series=rf_oos,
    mktrf_series=mktrf_oos,
    title=f"OUT-OF-SAMPLE: Long-Short Market Neutral ({dates_out_sample.min().date()} to {dates_out_sample.max().date()})",
    out_path="out/oos_long_short_tearsheet.html"
)